# Evaluación de SAN en CROHME 2016 y 2019 (Modelo2)

Este notebook carga el checkpoint congelado de **SAN** y lo prueba directamente con las imágenes y etiquetas de `data/Modelo2/{2016,2019}`. Calcula exactitud de expresión, tiempo de inferencia y conserva los errores por muestra.

> Para una prueba rápida, asigna un entero a `MAX_SAMPLES`. Con `None` se evalúan completos ambos conjuntos.

In [1]:
from pathlib import Path
import gc
import os
import sys
import time

import cv2
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'SAN' / '14.yaml').is_file() and (candidate / 'data' / 'Modelo2').is_dir():
            return candidate
    raise FileNotFoundError('No se encontró la raíz que contiene SAN/ y data/Modelo2/.')


ROOT = find_project_root()
SAN_DIR = ROOT / 'SAN'
DATA_DIR = ROOT / 'data' / 'Modelo2'
OUT_DIR = ROOT / 'Cascada' / '04_evaluacion' / 'resultados'
YEARS = ('2016', '2019')
MAX_SAMPLES = None  # Ejemplo: 25 para una prueba rápida; None para el test completo.
SAVE_RESULTS = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for year in YEARS:
    assert (DATA_DIR / year / 'caption.txt').is_file(), f'Faltan etiquetas de {year}'
    assert (DATA_DIR / year / 'img').is_dir(), f'Faltan imágenes de {year}'
assert (SAN_DIR / 'checkpoints' / 'SAN_decoder' / 'best.pth').is_file(), 'Falta el checkpoint de SAN'

print(f'Raíz: {ROOT}')
print(f'Dispositivo: {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))
print('Muestras:', {year: sum(1 for line in (DATA_DIR / year / 'caption.txt').open(encoding='utf-8') if line.strip()) for year in YEARS})

/home/mora/miniconda3/envs/hmer-gpu/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raíz: /mnt/c/Users/novac/Documents/Unidad/Documentos/IPN/Trabajo_terminal/Programas
Dispositivo: cuda (NVIDIA GeForce RTX 4050 Laptop GPU)
Muestras: {'2016': 1147, '2019': 1199}


In [2]:
# SAN usa rutas relativas en 14.yaml; por eso se carga temporalmente desde SAN_DIR.
sys.path.insert(0, str(SAN_DIR))
from dataset import Words
from infer.Backbone import Backbone
from utils import load_checkpoint, load_config

previous_cwd = Path.cwd()
try:
    os.chdir(SAN_DIR)
    params = load_config('14.yaml')
    params['device'] = DEVICE
    words = Words(params['word_path'])
    params.update(word_num=len(words), struct_num=7, words=words)
    model = Backbone(params).to(DEVICE)
    load_checkpoint(model, None, params['checkpoint'])
finally:
    os.chdir(previous_cwd)

model.eval()
model.requires_grad_(False)
print(f'Checkpoint cargado; vocabulario: {len(words)} símbolos.')

115 symbols in total
No optimizer in the pretrained model
Checkpoint cargado; vocabulario: 115 símbolos.


In [3]:
def read_captions(path):
    captions = {}
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            if not line.strip():
                continue
            sample_id, label = line.rstrip('\r\n').split('\t', 1)
            if sample_id in captions:
                raise ValueError(f'ID repetido: {sample_id}')
            captions[sample_id] = ' '.join(label.split())
    return captions


def san_to_latex(prediction):
    """Convierte el árbol de salida de SAN a la secuencia LaTeX usada por CROHME."""
    if prediction is None or len(prediction) <= 1:
        return '<sin predicción>'

    children = {}
    for item in prediction:
        if len(item) < 4:
            continue
        try:
            node_id, parent_id = int(item[1]), int(item[2])
        except (TypeError, ValueError):
            continue
        children.setdefault(parent_id, []).append((node_id, str(item[3])))

    def walk(node_id):
        if not 0 <= node_id < len(prediction):
            return ['<nodo inválido>']
        symbol = str(prediction[node_id][0])
        kids = children.get(node_id, [])
        if not kids:
            return [symbol]
        result = [symbol]
        if symbol == r'\frac':
            for relation in ('Above', 'Below'):
                for child_id, child_relation in kids:
                    if child_relation == relation:
                        result += ['{'] + walk(child_id) + ['}']
            for child_id, relation in kids:
                if relation == 'Right':
                    result += walk(child_id)
            return result
        for child_id, relation in kids:
            if relation == 'l_sup':
                result += ['['] + walk(child_id) + [']']
        for child_id, relation in kids:
            if relation == 'Inside':
                result += ['{'] + walk(child_id) + ['}']
        for child_id, relation in kids:
            if relation in ('Sub', 'Below'):
                result += ['_', '{'] + walk(child_id) + ['}']
        for child_id, relation in kids:
            if relation in ('Sup', 'Above'):
                result += ['^', '{'] + walk(child_id) + ['}']
        for child_id, relation in kids:
            if relation == 'Right':
                result += walk(child_id)
        return result

    return ' '.join(walk(1))


def without_limits(text):
    return ' '.join(token for token in text.split() if token != r'\limits')

In [4]:
def evaluate_year(year, max_samples=None):
    captions = read_captions(DATA_DIR / year / 'caption.txt')
    sample_ids = list(captions)
    if max_samples is not None:
        if max_samples <= 0:
            raise ValueError('MAX_SAMPLES debe ser positivo o None.')
        sample_ids = sample_ids[:max_samples]

    rows = []
    with torch.inference_mode():
        for sample_id in tqdm(sample_ids, desc=year):
            image_path = DATA_DIR / year / 'img' / f'{sample_id}.bmp'
            row = {'year': year, 'sample_id': sample_id, 'ground_truth': captions[sample_id]}
            try:
                image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
                if image is None:
                    raise FileNotFoundError(image_path)
                tensor = torch.as_tensor(image, dtype=torch.float32, device=DEVICE)[None, None] / 255.0
                mask = torch.ones_like(tensor)
                if DEVICE.type == 'cuda':
                    torch.cuda.synchronize()
                start = time.perf_counter()
                prediction = model(tensor, mask)
                if DEVICE.type == 'cuda':
                    torch.cuda.synchronize()
                elapsed_ms = (time.perf_counter() - start) * 1000
                latex = san_to_latex(prediction)
                row.update(
                    status='ok',
                    error='',
                    height=image.shape[0],
                    width=image.shape[1],
                    prediction=latex,
                    exact=int(latex == captions[sample_id]),
                    exact_no_limits=int(without_limits(latex) == without_limits(captions[sample_id])),
                    time_ms=elapsed_ms,
                )
            except Exception as exc:
                row.update(status='error', error=repr(exc), height=None, width=None,
                           prediction='', exact=0, exact_no_limits=0, time_ms=None)
                if 'out of memory' in str(exc).lower():
                    raise
            rows.append(row)
    return pd.DataFrame(rows)

In [5]:
results = {}
for year in YEARS:
    results[year] = evaluate_year(year, MAX_SAMPLES)
    if SAVE_RESULTS:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        suffix = 'completo' if MAX_SAMPLES is None else f'n{MAX_SAMPLES}'
        output_path = OUT_DIR / f'san_modelo2_{year}_{suffix}.csv'
        results[year].to_csv(output_path, index=False, encoding='utf-8')
        print(f'Guardado: {output_path}')
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

2016:   0%|          | 0/1147 [00:00<?, ?it/s]/home/mora/miniconda3/envs/hmer-gpu/lib/python3.8/site-packages/torch/nn/modules/conv.py:459: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,
2016: 100%|██████████| 1147/1147 [01:58<00:00,  9.67it/s]


Guardado: /mnt/c/Users/novac/Documents/Unidad/Documentos/IPN/Trabajo_terminal/Programas/Cascada/04_evaluacion/resultados/san_modelo2_2016_completo.csv


2019: 100%|██████████| 1199/1199 [01:28<00:00, 13.59it/s]


Guardado: /mnt/c/Users/novac/Documents/Unidad/Documentos/IPN/Trabajo_terminal/Programas/Cascada/04_evaluacion/resultados/san_modelo2_2019_completo.csv


In [6]:
summary_rows = []
for year, frame in results.items():
    valid = frame[frame['status'] == 'ok']
    summary_rows.append({
        'año': year,
        'muestras': len(frame),
        'válidas': len(valid),
        'errores': int((frame['status'] == 'error').sum()),
        'ExpRate': valid['exact'].mean() if len(valid) else float('nan'),
        'ExpRate_sin_limits': valid['exact_no_limits'].mean() if len(valid) else float('nan'),
        'latencia_media_ms': valid['time_ms'].mean() if len(valid) else float('nan'),
    })

summary = pd.DataFrame(summary_rows)
display(summary.style.format({
    'ExpRate': '{:.2%}',
    'ExpRate_sin_limits': '{:.2%}',
    'latencia_media_ms': '{:.2f}',
}))

for year, frame in results.items():
    print(f'\nEjemplos de {year}:')
    display(frame[['sample_id', 'ground_truth', 'prediction', 'exact', 'time_ms']].head(5))
    errors = frame[frame['status'] == 'error']
    if not errors.empty:
        print(f'Errores de {year}:')
        display(errors[['sample_id', 'error']])

,año,muestras,válidas,errores,ExpRate,ExpRate_sin_limits,latencia_media_ms
0,2016,1147,1147,0,51.00%,53.62%,92.05
1,2019,1199,1199,0,51.46%,54.30%,62.16



Ejemplos de 2016:


,sample_id,ground_truth,prediction,exact,time_ms
0,UN_131_em_1083,[ e ],[ e ],1,898.662694
1,UN_133_em_1133,A . A,A . A,1,129.727704
2,UN_454_em_695,3 . 8,3 . 8,1,113.934246
3,UN_465_em_965,F . G,F,0,103.987286
4,UN_460_em_831,7 + 7,7 + 7,1,140.529585



Ejemplos de 2019:


,sample_id,ground_truth,prediction,exact,time_ms
0,UN19_1013_em_184,t r,t _ { c },0,101.292839
1,UN19_1019_em_266,I H,I H,1,14.071114
2,UN19_1045_em_658,I I,I I,1,15.315503
3,UN19wb_1119_em_1168,h h,h h,1,76.118087
4,UN19_1022_em_307,2 m,2 m,1,16.320793
